# Les poupées russes de la monnaie · *The nesting dolls of money*

Notebook compagnon du chapitre **25. Masse monétaire M1, M2 : ce que ces agrégats mesurent vraiment** — [lire l'article](https://nmlab.io/ressources/masse-monetaire-m1-m2).
Companion notebook to chapter **25. Money Supply M1, M2: What These Aggregates Really Measure** — [read the article](https://nmlab.io/en/ressources/money-supply-m1-m2).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure est régénérée par le code — un **schéma éditable** : changez les libellés à votre guise. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure from code — an **editable diagram**: change the labels as you like; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# (schéma : aucune donnée externe)


import numpy as np
import pandas as pd
from matplotlib.figure import Figure
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

C, W = nm.COLORS, nm.WIDTH_PX

EA_M3_KEY = "BSI/M.U2.Y.V.M30.X.1.U2.2300.Z01.E"   # encours M3 zone euro (BCE)


def load(series_id: str, start: str | None = None, end: str | None = None) -> pd.Series:
    """Charge une série en direct : FRED, ou le portail de la BCE pour « EA_M3 »."""
    if series_id == "EA_M3":
        url = (f"https://data-api.ecb.europa.eu/service/data/{EA_M3_KEY}"
               "?format=csvdata&detail=dataonly")
        raw = pd.read_csv(url)
        s = pd.Series(raw["OBS_VALUE"].values,
                      index=pd.PeriodIndex(raw["TIME_PERIOD"], freq="M").to_timestamp())
        s = s.sort_index() / 1000.0                 # millions -> milliards d'euros
    else:
        s = nm.load_fred(series_id)
    return s.loc[start:end]


def T(d: dict, lang: str):
    """Sélectionne le jeu de libellés de la langue demandée."""
    return d[lang]


def build_figure(lang: str = "fr") -> Figure:
    """Construit la figure NMLab du chapitre (libellés selon ``lang``)."""
    fig = nm.figure(1150); ax = nm.blank_axes(fig)
    d = dict(fr=("Des poupées russes de liquidité","Les agrégats du public s'emboîtent ; la base est une mesure à part.",
                 [("M3","le plus large","",C["edge"]),
                  ("M2","+ épargne courte, fonds monétaires","23,1 bn $ · 16,4 bn €",C["teal"]),
                  ("M1","numéraire + dépôts à vue (+ épargne aux É.-U.)","19,8 bn $ · 11,3 bn €",C["blue"])],
                 ("Base monétaire","billets + réserves","5,5 bn $","monnaie centrale"),
                 "Le numéraire (billets) est la seule part commune à la base et à M1 : la base n'est pas une poupée de plus.\nMontants États-Unis et zone euro, mai 2026 (bn = billions). Sources : FRED, BCE."),
             en=("Russian dolls of liquidity","Public money nests together; the base is a separate measure.",
                 [("M3","the broadest","",C["edge"]),
                  ("M2","+ short savings, money funds","$23.1 tn · €16.4 tn",C["teal"]),
                  ("M1","currency + demand deposits (+ savings in the U.S.)","$19.8 tn · €11.3 tn",C["blue"])],
                 ("Monetary base","banknotes + reserves","$5.5 tn","central-bank money"),
                 "Currency (banknotes) is the only part shared by the base and M1: the base is not one more doll.\nAmounts United States and euro area, May 2026 (tn = trillion). Sources: FRED, ECB."))
    t=T(d,lang); nm.header(fig,t[0],t[1])
    # Poupées russes M1 ⊂ M2 ⊂ M3 (à droite)
    cx=W*0.635; base_bottom=250
    widths=[900,680,450]; heights=[720,540,360]
    for i,(w,h,(lab,desc,amt,col)) in enumerate(zip(widths,heights,t[2])):
        x=cx-w/2; y=base_bottom
        nm.card(ax,x,y,w,h,edge=col,lw=2.6,radius=20,fill=C["bg"])
        ytop=y+h
        ax.text(cx, ytop-48, lab, ha="center", va="center", fontsize=30 if i==0 else 27,
                fontweight="bold", color=col)
        if desc: ax.text(cx, ytop-98, desc, ha="center", va="center", fontsize=16, color=C["muted"])
        if amt:  ax.text(cx, ytop-134, amt, ha="center", va="center", fontsize=17, color=C["text"], fontweight="bold")
    # Base monétaire, encadré séparé (à gauche)
    blab,bdesc,bamt,bsub=t[3]
    bx,by,bw,bh=70,250,430,430
    nm.card(ax,bx,by,bw,bh,edge=C["amber"],lw=2.8,radius=20,fill=C["card"])
    bcx=bx+bw/2
    ax.text(bcx,by+bh-70,blab,ha="center",va="center",fontsize=25,fontweight="bold",color=C["amber"])
    ax.text(bcx,by+bh-140,bdesc,ha="center",va="center",fontsize=17,color=C["muted"])
    ax.text(bcx,by+bh-205,bamt,ha="center",va="center",fontsize=20,color=C["text"],fontweight="bold")
    ax.text(bcx,by+70,bsub,ha="center",va="center",fontsize=16,color=C["muted"],style="italic")
    # séparateur : la base est distincte des poupées (pas de flèche traversante)
    ax.plot([bx+bw+55,bx+bw+55],[by+40,by+bh-40],color=C["edge"],lw=1.6,ls=(0,(4,4)))
    nm.footer(fig,t[4]);
    return fig


fig = build_figure(LANG)